# Snooker Ball Detection - Fine-tune YOLOv8
Fine-tune `best_color_old.pt` with our labeled dataset.

**Setup:**
1. Enable GPU: Settings → Accelerator → GPU T4 x2
2. Upload `kaggle_dataset.zip` and `best_color_old.pt` as Kaggle Datasets

In [ ]:
!pip install -q ultralytics

In [ ]:
import torch
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')

In [ ]:
# Find and prepare the dataset
import zipfile, os, shutil, glob

dataset_dir = '/kaggle/working/dataset'
if os.path.exists(dataset_dir):
    shutil.rmtree(dataset_dir)

# Search for the uploaded data in /kaggle/input/
input_root = '/kaggle/input'
found = False

# Case 1: It's a zip file
for zf in glob.glob(f'{input_root}/**/*.zip', recursive=True):
    print(f'Found zip: {zf}')
    with zipfile.ZipFile(zf, 'r') as z:
        z.extractall(dataset_dir)
    found = True
    break

# Case 2: It's an extracted folder with images/ and labels/ inside
if not found:
    for root, dirs, files in os.walk(input_root):
        if 'images' in dirs and 'labels' in dirs:
            print(f'Found dataset folder: {root}')
            shutil.copytree(root, dataset_dir)
            found = True
            break

# Case 3: The folder contains train/images, train/labels structure
if not found:
    for root, dirs, files in os.walk(input_root):
        if 'train' in dirs:
            train_path = os.path.join(root, 'train')
            if os.path.isdir(os.path.join(train_path, 'images')):
                print(f'Found dataset folder: {root}')
                shutil.copytree(root, dataset_dir)
                found = True
                break

if not found:
    # Show what's in /kaggle/input/ so we can debug
    print("Could not find dataset. Contents of /kaggle/input/:")
    for root, dirs, files in os.walk(input_root):
        level = root.replace(input_root, '').count(os.sep)
        if level < 4:
            indent = '  ' * level
            print(f'{indent}{os.path.basename(root)}/')
            for f in files[:5]:
                print(f'{indent}  {f}')
            if len(files) > 5:
                print(f'{indent}  ... +{len(files)-5} more')
    raise FileNotFoundError("Upload kaggle_dataset.zip or the dataset folder as a Kaggle Dataset")

# Count
train_imgs = glob.glob(f'{dataset_dir}/**/train/images/*', recursive=True)
val_imgs = glob.glob(f'{dataset_dir}/**/val/images/*', recursive=True)
if not train_imgs:
    train_imgs = glob.glob(f'{dataset_dir}/images/train/*')
    val_imgs = glob.glob(f'{dataset_dir}/images/val/*')
print(f'Train: {len(train_imgs)} images, Val: {len(val_imgs)} images')

In [ ]:
# Find actual dataset structure and fix data.yaml
import yaml, glob, os

# Show what's actually in the dataset dir
print("Dataset contents:")
for root, dirs, files in os.walk('/kaggle/working/dataset'):
    level = root.replace('/kaggle/working/dataset', '').count(os.sep)
    if level < 4:
        indent = '  ' * level
        print(f'{indent}{os.path.basename(root)}/')
        for f in files[:3]:
            print(f'{indent}  {f}')
        if len(files) > 3:
            print(f'{indent}  ... +{len(files)-3} more')

# Find where images/train actually lives
train_dirs = glob.glob('/kaggle/working/dataset/**/images/train', recursive=True)
val_dirs = glob.glob('/kaggle/working/dataset/**/images/val', recursive=True)

if train_dirs:
    ds_root = os.path.dirname(os.path.dirname(train_dirs[0]))
    print(f'\nDataset root: {ds_root}')
    print(f'Train dir: {train_dirs[0]} ({len(os.listdir(train_dirs[0]))} files)')
    if val_dirs:
        print(f'Val dir: {val_dirs[0]} ({len(os.listdir(val_dirs[0]))} files)')
    else:
        # No val dir — create one by moving 15% of train
        print('No val split found — creating one...')
        import random, shutil
        train_dir = train_dirs[0]
        label_train = train_dir.replace('/images/', '/labels/')
        val_img_dir = train_dir.replace('/train', '/val')
        val_lbl_dir = label_train.replace('/train', '/val')
        os.makedirs(val_img_dir, exist_ok=True)
        os.makedirs(val_lbl_dir, exist_ok=True)

        imgs = sorted(os.listdir(train_dir))
        random.seed(42)
        random.shuffle(imgs)
        val_count = max(1, int(len(imgs) * 0.15))
        for img in imgs[:val_count]:
            shutil.move(os.path.join(train_dir, img), os.path.join(val_img_dir, img))
            lbl = os.path.splitext(img)[0] + '.txt'
            lbl_src = os.path.join(label_train, lbl)
            if os.path.exists(lbl_src):
                shutil.move(lbl_src, os.path.join(val_lbl_dir, lbl))
        print(f'Moved {val_count} images to val')
else:
    ds_root = '/kaggle/working/dataset'
    print('WARNING: Could not find images/train directory!')

# Write data.yaml at the correct location
yaml_path = os.path.join(ds_root, 'data.yaml')
data = {
    'path': ds_root,
    'train': 'images/train',
    'val': 'images/val',
    'nc': 9,
    'names': ['black-ball', 'blue-ball', 'brown-ball', 'green-ball',
              'pink-ball', 'pocket', 'red-ball', 'white-ball', 'yellow-ball'],
}
with open(yaml_path, 'w') as f:
    yaml.dump(data, f)

print(f'\ndata.yaml written to: {yaml_path}')
print(f'Classes: {data["names"]}')

In [ ]:
# Find the base model (best_color_old.pt / best_color.pt)
import glob

model_candidates = glob.glob('/kaggle/input/**/*.pt', recursive=True)

if model_candidates:
    base_model = model_candidates[0]
    print(f'Base model: {base_model}')
else:
    base_model = 'yolov8s.pt'
    print(f'No uploaded model found, using {base_model} (training from scratch)')

# Fine-tune: freeze backbone (first 10 layers), gentle learning rate
from ultralytics import YOLO

print(f'Using data.yaml: {yaml_path}')
model = YOLO(base_model)
results = model.train(
    data=yaml_path,
    epochs=50,
    imgsz=640,
    batch=32,
    device=0,
    patience=15,
    workers=4,
    freeze=10,
    lr0=0.0005,
    save=True,
    plots=True,
    verbose=True,
)
print('Training complete!')

In [ ]:
# Show training results
from IPython.display import Image, display
import glob

for img in ['results.png', 'confusion_matrix.png', 'val_batch0_pred.jpg']:
    paths = glob.glob(f'runs/detect/train*/{img}')
    if paths:
        display(Image(filename=sorted(paths)[-1], width=800))

In [ ]:
# Validate best model
best_pt = sorted(glob.glob('runs/detect/train*/weights/best.pt'))[-1]
print(f'Best model: {best_pt}')

model = YOLO(best_pt)
metrics = model.val(data=yaml_path)
print(f'\nmAP50: {metrics.box.map50:.3f}')
print(f'mAP50-95: {metrics.box.map:.3f}')

In [ ]:
# Save model for download
import shutil
shutil.copy(best_pt, '/kaggle/working/best_color.pt')
print('Model saved to /kaggle/working/best_color.pt')
print()
print('Download it and copy to:')
print('  final/src/snookervision/data/model/best_color.pt')